## Setup and Configuration

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import ciw

# Repository imports
root_dir = Path("../../..").resolve()
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from hybridsim import des_component as des
from hybridsim import results as res

# Apply the custom Ciw record changes once, before constructing simulations.
des.apply_custom_record_changes()

In [2]:
# Model dimensions
SEVERITIES = ["Low", "Medium", "High"]

SUBSPECIALTIES = [
    "Foot/Ankle",
    "Hand",
    "Hip",
    "Knee",
    "Paeds",
    "Shoulder/Elbow",
    "Spine",
]

ALPHABET = ["A", "B", "C", "D", "E", "F", "G"]
EMERGENCY_NODES = ["B", "E", "G"]

PRE_OP_LETTER = "C"
ELECTIVE_SURGERY_LETTER = "D"

ACTIVITIES_TO_REPORT = ["A", "C", "D", "F"]

ACTIVITY_NAMES = {
    "A": "Elective new outpatient",
    "B": "Non-elective new outpatient",
    "C": "Pre-operative assessment",
    "D": "Elective admission / surgery",
    "E": "Non-elective admission / surgery",
    "F": "Elective follow-up outpatient",
    "G": "Non-elective follow-up outpatient",
    "*": "Referral node",
}

# All toy-model patients are allocated to Foot/Ankle.
FOOT_ANKLE_ONLY_PROBABILITIES = [1, 0, 0, 0, 0, 0, 0]

SUBSPECIALTY_PROBABILITIES = {
    severity: FOOT_ANKLE_ONLY_PROBABILITIES.copy()
    for severity in SEVERITIES
}

# Every scenario starts with an empty system and runs for one year.
RUN_TIME_DAYS = 365
SCENARIO_SEED = 0
WARMUP_DAYS = 0

OUTPUT_DIR = Path("../outputs/des_behaviour").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The order here is retained throughout the output tables.
SCENARIO_DEFINITIONS = [
    {
        "scenario_index": 0,
        "scenario_key": "no_arrivals",
        "scenario_name": "No arrivals",
        "arrival_rates": {
            "Low": 0.0,
            "Medium": 0.0,
            "High": 0.0,
        },
    },
    {
        "scenario_index": 1,
        "scenario_key": "below_capacity",
        "scenario_name": "Below capacity",
        "arrival_rates": {
            "Low": 0.2,
            "Medium": 0.25,
            "High": 0.3,
        },
    },
    {
        "scenario_index": 2,
        "scenario_key": "activity_d_overloaded",
        "scenario_name": "Activity D overloaded",
        "arrival_rates": {
            "Low": 0.2,
            "Medium": 0.6,
            "High": 0.7,
        },
    },
    {
        "scenario_index": 3,
        "scenario_key": "severe_overload",
        "scenario_name": "Severe overload",
        "arrival_rates": {
            "Low": 2.0,
            "Medium": 2.0,
            "High": 2.0,
        },
    },
]

SCENARIO_ORDER = [
    scenario["scenario_name"]
    for scenario in SCENARIO_DEFINITIONS
]

# Number of visits to each reported activity under each toy PDFA.
PATHWAY_VISIT_COUNTS = pd.DataFrame(
    {
        "A": [1, 1, 1],
        "C": [0, 1, 1],
        "D": [0, 1, 1],
        "F": [1, 0, 2],
    },
    index=SEVERITIES,
)

## Toy PDFA Construction

In [4]:
def build_sequential_pdfa(alphabet, sequence):
    """Build a deterministic PDFA for a known activity sequence."""
    n_states = len(sequence) + 2
    matrix = np.zeros(
        (len(alphabet), n_states, n_states),
        dtype=float,
    )

    # Initial state transition used by the DES routing implementation.
    matrix[0, 0, 1] = 1.0

    for step, activity in enumerate(sequence, start=1):
        activity_index = alphabet.index(activity)
        matrix[activity_index, step, step + 1] = 1.0

    return matrix


def build_probabilistic_pdfa(alphabet, transitions, n_states):
    """
    Build a small hand-coded PDFA from explicit transitions.

    Each transition is supplied as:
        (from_state, activity_letter, to_state, probability)
    """
    matrix = np.zeros(
        (len(alphabet), n_states, n_states),
        dtype=float,
    )

    for from_state, activity, to_state, probability in transitions:
        activity_index = alphabet.index(activity)
        matrix[
            activity_index,
            from_state,
            to_state,
        ] = probability

    return matrix

In [5]:
activity_dict, inverted_activity_dict = (
    des.get_activity_dictionaries(
        ALPHABET,
        start_value=3,
    )
)

subspec_dict = {
    subspecialty: index
    for index, subspecialty in enumerate(SUBSPECIALTIES)
}

# Deterministic severity-specific pathways.
low_pdfa = build_sequential_pdfa(
    ALPHABET,
    ["A", "F"],
)
medium_pdfa = build_sequential_pdfa(
    ALPHABET,
    ["A", "C", "D"],
)
high_pdfa = build_sequential_pdfa(
    ALPHABET,
    ["A", "C", "D", "F", "F"],
)

# The DES module expects Low, Medium, and High matrices for each
# subspecialty in turn.
pdfas = [
    pdfa
    for _ in SUBSPECIALTIES
    for pdfa in [low_pdfa, medium_pdfa, high_pdfa]
]

alphabets = [
    ALPHABET
    for _ in pdfas
]

nodes = des.get_list_of_nodes(
    alphabets,
    SUBSPECIALTIES,
)

print("Activity dictionary:", activity_dict)

Activity dictionary: {'A': 3, 'B': 4, 'C': 5, 'D': 6, 'E': 7, 'F': 8, 'G': 9}


## Service Distributions

In [ ]:
def make_deterministic_service_distributions(service_times):
    """
    Convert one service-time value per activity into Ciw distributions.

    The values must follow the order given by `ALPHABET`.
    """
    if len(service_times) != len(ALPHABET):
        raise ValueError(
            "One service time must be supplied for every activity "
            f"in ALPHABET ({len(ALPHABET)} values expected)."
        )

    return [
        ciw.dists.Deterministic(value=value)
        for value in service_times
    ]


# Foot/Ankle values retained from the original notebook.
FOOT_ANKLE_SERVICE_TIMES = [
    365 / 1220,
    0,
    365 / 585,
    365 / 458.9,
    0,
    365 / 1280,
    0,
]

# Placeholder values used for the remaining subspecialties. They do
# not affect the current toy scenarios because all patients are
# allocated to Foot/Ankle.
PLACEHOLDER_SERVICE_TIMES = [
    1,
    0,
    1,
    1,
    0,
    1,
    0,
]

SERVICE_TIMES_BY_SUBSPECIALTY = {
    "Foot/Ankle": FOOT_ANKLE_SERVICE_TIMES,
    "Hand": PLACEHOLDER_SERVICE_TIMES,
    "Hip": PLACEHOLDER_SERVICE_TIMES,
    "Knee": PLACEHOLDER_SERVICE_TIMES,
    "Paeds": PLACEHOLDER_SERVICE_TIMES,
    "Shoulder/Elbow": PLACEHOLDER_SERVICE_TIMES,
    "Spine": PLACEHOLDER_SERVICE_TIMES,
}

subspecialty_service_dists = [
    make_deterministic_service_distributions(
        SERVICE_TIMES_BY_SUBSPECIALTY[subspecialty]
    )
    for subspecialty in SUBSPECIALTIES
]

# One server is used at each activity. The nominal daily capacity is
# therefore the reciprocal of the deterministic service time.
ACTIVITY_SERVICE_TIMES = {
    activity: FOOT_ANKLE_SERVICE_TIMES[ALPHABET.index(activity)]
    for activity in ACTIVITIES_TO_REPORT
}

ACTIVITY_CAPACITIES = {
    activity: 1.0 / service_time
    for activity, service_time in ACTIVITY_SERVICE_TIMES.items()
}

activity_capacity_df = pd.DataFrame(
    {
        "activity_letter": ACTIVITIES_TO_REPORT,
        "activity_name": [
            ACTIVITY_NAMES[activity]
            for activity in ACTIVITIES_TO_REPORT
        ],
        "service_time_days": [
            ACTIVITY_SERVICE_TIMES[activity]
            for activity in ACTIVITIES_TO_REPORT
        ],
        "capacity_per_day": [
            ACTIVITY_CAPACITIES[activity]
            for activity in ACTIVITIES_TO_REPORT
        ],
    }
)

activity_capacity_df

,activity_letter,activity_name,service_time_days,capacity_per_day
0,A,Elective new outpatient,0.299180,3.342466
1,C,Pre-operative assessment,0.623932,1.602740
2,D,Elective admission / surgery,0.795380,1.257260
3,F,Elective follow-up outpatient,0.285156,3.506849


## Routing and Reneging Objects

In [7]:
# Base PDFA routing object retained for direct inspection if required.
pdfa_routing = des.PDFARouting(
    pdfa_matrices=pdfas,
    alphabets=alphabets,
    activity_dict=activity_dict,
    subspec_dict=subspec_dict,
    pre_op_letter=PRE_OP_LETTER,
    elective_surgery_letter=ELECTIVE_SURGERY_LETTER,
)

# The toy network below uses the jockeying-enabled routing object.
jockeying_routing = des.JockeyRouting(
    pdfa_matrix=pdfas,
    alphabet=alphabets,
    activity_dict=activity_dict,
    subspec_dict=subspec_dict,
    pre_op_letter=PRE_OP_LETTER,
    elective_surgery_letter=ELECTIVE_SURGERY_LETTER,
)

pre_op_expiry_distribution = des.PreOpExpiryDist(
    activity_dict=activity_dict,
    subspec_dict=subspec_dict,
    pre_op_letter=PRE_OP_LETTER,
    elective_surgery_letter=ELECTIVE_SURGERY_LETTER,
)

## Reusable Scenario Functions

In [8]:
def deterministic_interarrival(rate):
    """Return a deterministic inter-arrival distribution for a daily rate."""
    if rate is None or rate <= 0:
        return None

    return ciw.dists.Deterministic(
        value=1.0 / rate,
    )


def make_deterministic_gp_arrivals(arrival_rates):
    """Create deterministic GP arrivals in Low, Medium, High order."""
    return [
        deterministic_interarrival(
            arrival_rates.get(severity, 0.0)
        )
        for severity in SEVERITIES
    ]


def build_toy_network(
    arrival_rates,
    *,
    service_distributions=None,
    emergency_nodes=None,
    routing_object=None,
    reneging_distribution=None,
):
    """Construct the toy DES network for a set of severity arrival rates."""
    if service_distributions is None:
        service_distributions = subspecialty_service_dists

    if emergency_nodes is None:
        emergency_nodes = EMERGENCY_NODES

    if routing_object is None:
        routing_object = jockeying_routing

    if reneging_distribution is None:
        reneging_distribution = pre_op_expiry_distribution

    gp_arrival_rates = make_deterministic_gp_arrivals(
        arrival_rates
    )

    return des.get_network(
        alphabets=alphabets,
        subspecialties=SUBSPECIALTIES,
        subspecialty_service_dists=service_distributions,
        emergency_nodes=emergency_nodes,
        subspecialty_class=routing_object,
        reneging_distribution=reneging_distribution,
        subspec_probs_low=SUBSPECIALTY_PROBABILITIES["Low"],
        subspec_probs_medium=SUBSPECIALTY_PROBABILITIES["Medium"],
        subspec_probs_high=SUBSPECIALTY_PROBABILITIES["High"],
        gp_arrival_rates=gp_arrival_rates,
        other_arrival_rates=[None, None, None],
    )

In [9]:
def get_server_utilisation_table(simulation):
    """Return server utilisation in a readable table."""
    utilisation = pd.DataFrame(
        {
            "node": range(
                1,
                len(simulation.transitive_nodes) + 1,
            ),
            "server_utilisation": [
                node.server_utilisation
                for node in simulation.transitive_nodes
            ],
        }
    )

    utilisation["activity_letter"] = utilisation["node"].map(
        inverted_activity_dict
    )
    utilisation["activity_name"] = utilisation[
        "activity_letter"
    ].map(ACTIVITY_NAMES)

    return utilisation


def run_toy_scenario(
    scenario_name,
    arrival_rates,
    *,
    scenario_key=None,
    scenario_index=None,
    run_time=RUN_TIME_DAYS,
    seed=SCENARIO_SEED,
    service_distributions=None,
    emergency_nodes=None,
    routing_object=None,
    reneging_distribution=None,
):
    """Build and run one deterministic toy DES scenario."""
    ciw.seed(seed)

    network = build_toy_network(
        arrival_rates=arrival_rates,
        service_distributions=service_distributions,
        emergency_nodes=emergency_nodes,
        routing_object=routing_object,
        reneging_distribution=reneging_distribution,
    )

    simulation = ciw.Simulation(network)
    simulation.simulate_until_max_time(run_time)

    records = pd.DataFrame(
        simulation.get_all_records()
    )

    # Add scenario metadata even when the record table is empty.
    records = records.assign(
        scenario_index=scenario_index,
        scenario_key=scenario_key,
        scenario=scenario_name,
        seed=seed,
        run_time_days=run_time,
    )

    return {
        "simulation": simulation,
        "records": records,
        "server_utilisation": get_server_utilisation_table(
            simulation
        ),
    }


def summarise_toy_records(
    records,
    *,
    scenario_name,
    scenario_index,
    trial,
    seed,
    warmup_days=WARMUP_DAYS,
):
    """Apply warm-up removal and create the standard DES summaries."""
    if records.empty:
        return {
            "patient_records": records.copy(),
            "activity_records": records.copy(),
            "patient_summary": pd.DataFrame(),
            "cohort_summary": pd.DataFrame(),
            "activity_summary": pd.DataFrame(),
        }

    patient_records = res.remove_warmup_patients(
        records_df=records,
        warmup_days=warmup_days,
        reset_time=True,
    )

    activity_records = res.remove_warmup_activity_records(
        records_df=records,
        warmup_days=warmup_days,
        reset_time=True,
    )

    patient_df, cohort_df, activity_summary_df = (
        res.summarise_des_records(
            patient_records=patient_records,
            activity_records=activity_records,
            subspecialties=SUBSPECIALTIES,
            activity_dictionary=activity_dict,
            scenario_name=scenario_name,
            scenario_index=scenario_index,
            trial=trial,
            seed=seed,
            nodes=nodes,
        )
    )

    return {
        "patient_records": patient_records,
        "activity_records": activity_records,
        "patient_summary": patient_df,
        "cohort_summary": cohort_df,
        "activity_summary": activity_summary_df,
    }


def mean_waiting_time_by_activity(
    records,
    activity_letters=ACTIVITIES_TO_REPORT,
):
    """Summarise raw waiting times for the reported activity nodes."""
    selected_nodes = {
        activity_dict[letter]: letter
        for letter in activity_letters
    }

    complete_activity_list = pd.DataFrame(
        {
            "node": list(selected_nodes),
            "activity_letter": list(selected_nodes.values()),
        }
    )

    if records.empty:
        summary = complete_activity_list.assign(
            n_activity_records=0,
            mean_waiting_time=np.nan,
        )
    else:
        observed_summary = (
            records.loc[
                records["node"].isin(selected_nodes),
                ["node", "waiting_time"],
            ]
            .groupby("node", as_index=False)
            .agg(
                n_activity_records=("waiting_time", "size"),
                mean_waiting_time=("waiting_time", "mean"),
            )
        )

        summary = complete_activity_list.merge(
            observed_summary,
            on="node",
            how="left",
        )

        summary["n_activity_records"] = (
            summary["n_activity_records"]
            .fillna(0)
            .astype(int)
        )

    summary["activity_name"] = summary[
        "activity_letter"
    ].map(ACTIVITY_NAMES)

    return summary[
        [
            "node",
            "activity_letter",
            "activity_name",
            "n_activity_records",
            "mean_waiting_time",
        ]
    ]


def calculate_activity_loading(arrival_rates):
    """
    Calculate offered activity rates and nominal traffic intensities.

    Offered activity rates are obtained from the external severity
    arrival rates and the number of visits specified by each toy PDFA.
    """
    severity_arrivals = pd.Series(
        arrival_rates,
        dtype=float,
    ).reindex(SEVERITIES)

    offered_rates = PATHWAY_VISIT_COUNTS.mul(
        severity_arrivals,
        axis=0,
    ).sum(axis=0)

    loading = pd.DataFrame(
        {
            "activity_letter": ACTIVITIES_TO_REPORT,
            "activity_name": [
                ACTIVITY_NAMES[activity]
                for activity in ACTIVITIES_TO_REPORT
            ],
            "offered_arrival_rate": [
                offered_rates[activity]
                for activity in ACTIVITIES_TO_REPORT
            ],
            "service_capacity": [
                ACTIVITY_CAPACITIES[activity]
                for activity in ACTIVITIES_TO_REPORT
            ],
        }
    )

    loading["traffic_intensity"] = (
        loading["offered_arrival_rate"]
        / loading["service_capacity"]
    )

    return loading


def combine_nonempty_tables(tables):
    """Combine a list of result tables without failing on empty scenarios."""
    nonempty_tables = [
        table
        for table in tables
        if table is not None and not table.empty
    ]

    if not nonempty_tables:
        return pd.DataFrame()

    return pd.concat(
        nonempty_tables,
        ignore_index=True,
    )


def run_behavioural_scenarios(
    scenario_definitions=SCENARIO_DEFINITIONS,
):
    """
    Run, store, and summarise every DES behavioural scenario.

    Returns
    -------
    scenario_runs:
        Dictionary keyed by `scenario_key`, retaining each Ciw
        simulation and its scenario-specific result tables.
    combined_results:
        Dictionary containing combined tabular results across scenarios.
    """
    scenario_runs = {}

    scenario_input_tables = []
    raw_record_tables = []
    server_utilisation_tables = []
    loading_tables = []
    waiting_time_tables = []
    patient_summary_tables = []
    cohort_summary_tables = []
    activity_summary_tables = []

    for scenario in scenario_definitions:
        scenario_index = scenario["scenario_index"]
        scenario_key = scenario["scenario_key"]
        scenario_name = scenario["scenario_name"]
        arrival_rates = scenario["arrival_rates"]

        run = run_toy_scenario(
            scenario_name=scenario_name,
            scenario_key=scenario_key,
            scenario_index=scenario_index,
            arrival_rates=arrival_rates,
            run_time=RUN_TIME_DAYS,
            seed=SCENARIO_SEED,
        )

        summaries = summarise_toy_records(
            run["records"],
            scenario_name=scenario_name,
            scenario_index=scenario_index,
            trial=0,
            seed=SCENARIO_SEED,
        )

        loading = calculate_activity_loading(
            arrival_rates
        ).assign(
            scenario_index=scenario_index,
            scenario_key=scenario_key,
            scenario=scenario_name,
        )

        waiting_times = mean_waiting_time_by_activity(
            run["records"]
        ).assign(
            scenario_index=scenario_index,
            scenario_key=scenario_key,
            scenario=scenario_name,
        )

        server_utilisation = run[
            "server_utilisation"
        ].assign(
            scenario_index=scenario_index,
            scenario_key=scenario_key,
            scenario=scenario_name,
        )

        scenario_input_tables.append(
            pd.DataFrame(
                [
                    {
                        "scenario_index": scenario_index,
                        "scenario_key": scenario_key,
                        "scenario": scenario_name,
                        "lambda_low": arrival_rates["Low"],
                        "lambda_medium": arrival_rates["Medium"],
                        "lambda_high": arrival_rates["High"],
                        "run_time_days": RUN_TIME_DAYS,
                        "seed": SCENARIO_SEED,
                    }
                ]
            )
        )

        raw_record_tables.append(run["records"])
        server_utilisation_tables.append(server_utilisation)
        loading_tables.append(loading)
        waiting_time_tables.append(waiting_times)

        for table_name, table_collection in [
            ("patient_summary", patient_summary_tables),
            ("cohort_summary", cohort_summary_tables),
            ("activity_summary", activity_summary_tables),
        ]:
            table = summaries[table_name].copy()

            if not table.empty:
                table["scenario_key"] = scenario_key
                table_collection.append(table)

        scenario_runs[scenario_key] = {
            **run,
            **summaries,
            "scenario_definition": scenario.copy(),
            "activity_loading": loading,
            "waiting_times": waiting_times,
        }

    combined_results = {
        "scenario_inputs": combine_nonempty_tables(
            scenario_input_tables
        ),
        "raw_records": combine_nonempty_tables(
            raw_record_tables
        ),
        "server_utilisation": combine_nonempty_tables(
            server_utilisation_tables
        ),
        "activity_loading": combine_nonempty_tables(
            loading_tables
        ),
        "waiting_times": combine_nonempty_tables(
            waiting_time_tables
        ),
        "patient_summary": combine_nonempty_tables(
            patient_summary_tables
        ),
        "cohort_summary": combine_nonempty_tables(
            cohort_summary_tables
        ),
        "activity_summary": combine_nonempty_tables(
            activity_summary_tables
        ),
    }

    return scenario_runs, combined_results


def save_behavioural_results(
    combined_results,
    output_dir=OUTPUT_DIR,
):
    """Save all combined tabular outputs as reproducible CSV files."""
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    saved_files = []

    for result_name, result_table in combined_results.items():
        output_path = output_dir / f"{result_name}.csv"
        result_table.to_csv(
            output_path,
            index=False,
        )

        saved_files.append(
            {
                "result": result_name,
                "path": str(output_path),
                "n_rows": len(result_table),
            }
        )

    activity_capacity_path = (
        output_dir / "activity_capacities.csv"
    )
    activity_capacity_df.to_csv(
        activity_capacity_path,
        index=False,
    )

    saved_files.append(
        {
            "result": "activity_capacities",
            "path": str(activity_capacity_path),
            "n_rows": len(activity_capacity_df),
        }
    )

    return pd.DataFrame(saved_files)

## Run All Behavioural-Assessment Scenarios

The four scenarios are defined once in `SCENARIO_DEFINITIONS` and executed by the same batch runner. The returned `scenario_runs` dictionary retains each Ciw simulation and its scenario-specific outputs. The `combined_results` dictionary contains corresponding tables combined across all four scenarios.

In [10]:
scenario_runs, combined_results = (
    run_behavioural_scenarios()
)

scenario_inputs_df = combined_results[
    "scenario_inputs"
]

scenario_inputs_df

,scenario_index,scenario_key,scenario,lambda_low,lambda_medium,lambda_high,run_time_days,seed
0,0,no_arrivals,No arrivals,0.0,0.00,0.0,365,0
1,1,below_capacity,Below capacity,0.2,0.25,0.3,365,0
2,2,activity_d_overloaded,Activity D overloaded,0.2,0.60,0.7,365,0
3,3,severe_overload,Severe overload,2.0,2.00,2.0,365,0


### Basic Behavioural Checks

These checks confirm that the scenario definitions produce the intended nominal loading conditions.

In [11]:
traffic_intensity_df = (
    combined_results["activity_loading"]
    .pivot(
        index="scenario",
        columns="activity_letter",
        values="traffic_intensity",
    )
    .reindex(
        index=SCENARIO_ORDER,
        columns=ACTIVITIES_TO_REPORT,
    )
)

assert scenario_runs["no_arrivals"]["records"].empty

assert (
    traffic_intensity_df.loc[
        "Below capacity"
    ] < 1
).all()

assert (
    traffic_intensity_df.loc[
        "Activity D overloaded",
        ["A", "C", "F"],
    ] < 1
).all()

assert (
    traffic_intensity_df.loc[
        "Activity D overloaded",
        "D",
    ] > 1
)

assert (
    traffic_intensity_df.loc[
        "Severe overload"
    ] > 1
).all()

print("All intended loading-condition checks passed.")

All intended loading-condition checks passed.


## Traffic-Intensity

The offered activity rates are calculated from the external severity arrival rates and the visit counts implied by the three toy PDFAs. The nominal traffic intensity is then calculated as the offered rate divided by the one-server service capacity.

In [12]:
traffic_intensity_display = (
    traffic_intensity_df
    .loc[
        [
            "Below capacity",
            "Activity D overloaded",
            "Severe overload",
        ]
    ]
    .round(2)
)

traffic_intensity_display

activity_letter,A,C,D,F
scenario,,,,
Below capacity,0.22,0.34,0.44,0.23
Activity D overloaded,0.45,0.81,1.03,0.46
Severe overload,1.80,2.50,3.18,1.71


## Mean Waiting-Times

Waiting times are calculated directly from the Ciw activity records generated during each 365-day run. Activities not visited in the no-arrivals scenario remain missing and are displayed as dashes.

In [13]:
mean_waiting_time_df = (
    combined_results["waiting_times"]
    .pivot(
        index="scenario",
        columns="activity_letter",
        values="mean_waiting_time",
    )
    .reindex(
        index=SCENARIO_ORDER,
        columns=ACTIVITIES_TO_REPORT,
    )
)

mean_waiting_time_display = (
    mean_waiting_time_df
    .round(2)
    .fillna("--")
)

mean_waiting_time_display

activity_letter,A,C,D,F
scenario,,,,
No arrivals,--,--,--,--
Below capacity,0.08,0.02,0.04,0.04
Activity D overloaded,0.09,0.1,6.05,0.02
Severe overload,80.8,50.91,39.09,0.07


## Inspect Scenario-Specific Results

Each run remains available through its scenario key. For example, the cells below display server utilisation and the first raw activity records from the isolated-\(D\)-overload scenario.

In [14]:
scenario_runs[
    "activity_d_overloaded"
]["server_utilisation"]

,node,server_utilisation,activity_letter,activity_name
0,1,NaN,,NaN
1,2,NaN,NaN,NaN
2,3,0.446721,A,Elective new outpatient
3,4,NaN,B,Non-elective new outpatient
4,5,0.807975,C,Pre-operative assessment
5,6,0.993557,D,Elective admission / surgery
6,7,NaN,E,Non-elective admission / surgery
7,8,0.439063,F,Elective follow-up outpatient
8,9,NaN,G,Non-elective follow-up outpatient
9,10,0.000000,NaN,NaN


In [15]:
scenario_runs[
    "activity_d_overloaded"
]["records"].head()

,id_number,customer_class,original_customer_class,node,arrival_date,waiting_time,service_start_date,service_time,service_end_date,time_blocked,...,queue_size_at_departure,server_id,record_type,level,referral_source,scenario_index,scenario_key,scenario,seed,run_time_days
0,545,High,High,1,364.285714,0.000000,364.285714,0.000000,364.285714,0.0,...,0,False,service,High,GP,2,activity_d_overloaded,Activity D overloaded,0,365
1,545,Foot/Ankle,Foot/Ankle,3,364.285714,0.000000,364.285714,0.299180,364.584895,0.0,...,0,1,service,High,GP,2,activity_d_overloaded,Activity D overloaded,0,365
2,526,High,High,1,351.428571,0.000000,351.428571,0.000000,351.428571,0.0,...,0,False,service,High,GP,2,activity_d_overloaded,Activity D overloaded,0,365
3,526,Foot/Ankle,Foot/Ankle,3,351.428571,0.000000,351.428571,0.299180,351.727752,0.0,...,1,1,service,High,GP,2,activity_d_overloaded,Activity D overloaded,0,365
4,526,Foot/Ankle,Foot/Ankle,5,351.727752,0.118472,351.846224,0.623932,352.470156,0.0,...,1,1,service,High,GP,2,activity_d_overloaded,Activity D overloaded,0,365


The combined standard DES summaries are also available for further inspection:

In [16]:
combined_results["cohort_summary"].head()

,scenario,scenario_index,trial,seed,aggregation_level,subspecialty,severity,referral_source,n_patients,n_completed,...,median_total_wait_completed,p90_total_wait_completed,mean_total_service_all_patients,mean_total_service_completed,mean_n_completed_activities_all_patients,median_n_completed_activities_all_patients,mean_n_completed_activities_completed,median_n_completed_activities_completed,n_incomplete,scenario_key
0,Below capacity,1,0,0,Subspecialty x severity x referral source,Foot/Ankle,Low,GP,72,72,...,0.037801,0.29918,0.584337,0.584337,2.000000,2.0,2.0,2.0,0,below_capacity
1,Below capacity,1,0,0,Subspecialty x severity x referral source,Foot/Ankle,Medium,GP,91,90,...,0.000000,0.29918,1.709752,1.718492,2.989011,3.0,3.0,3.0,1,below_capacity
2,Below capacity,1,0,0,Subspecialty x severity x referral source,Foot/Ankle,High,GP,109,108,...,0.285156,0.29918,2.276275,2.288805,4.972477,5.0,5.0,5.0,1,below_capacity
3,Below capacity,1,0,0,Subspecialty x severity,Foot/Ankle,Low,All,72,72,...,0.037801,0.29918,0.584337,0.584337,2.000000,2.0,2.0,2.0,0,below_capacity
4,Below capacity,1,0,0,Subspecialty x severity,Foot/Ankle,Medium,All,91,90,...,0.000000,0.29918,1.709752,1.718492,2.989011,3.0,3.0,3.0,1,below_capacity


In [17]:
combined_results["activity_summary"].head()

,scenario,scenario_index,trial,seed,subspecialty,severity,activity_letter,n_records,n_patients,n_completed_activities,...,total_service_time,mean_service_time,median_service_time,mean_queue_at_arrival,median_queue_at_arrival,p90_queue_at_arrival,mean_queue_at_departure,median_queue_at_departure,p90_queue_at_departure,scenario_key
0,Below capacity,1,0,0,Foot/Ankle,Low,A,72,72,72,...,21.540984,0.299180,0.299180,0.319444,0.0,1.0,0.430556,0.0,1.0,below_capacity
1,Below capacity,1,0,0,Foot/Ankle,Low,C,0,0,0,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,below_capacity
2,Below capacity,1,0,0,Foot/Ankle,Low,D,0,0,0,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,below_capacity
3,Below capacity,1,0,0,Foot/Ankle,Low,F,72,72,72,...,20.531250,0.285156,0.285156,0.500000,0.5,1.0,0.500000,0.5,1.0,below_capacity
4,Below capacity,1,0,0,Foot/Ankle,Medium,A,91,91,91,...,27.225410,0.299180,0.299180,0.219780,0.0,1.0,0.175824,0.0,1.0,below_capacity
